In [7]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "D:/Github/note")
sys.path.insert(0, "D:/Github/Quant/projects/CTA/學弟策略")

from pathlib import Path
import time
import pandas as pd
from tqdm import tqdm

from module.get_info_FinMind import FinMindClient
from utils import (
    compact_daily_price_parquet,
    get_valid_stock_ids,
    infer_stock_ids_from_kbar_dir,
    load_or_build_above_ma_signal,
    load_or_refresh_daily_prices,
    read_kbar_ids,
    signal_row_to_stock_ids,
    write_filtered_from_full_kbar,
)

client = FinMindClient()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## get data

### config

In [8]:
start_date = "2022-01-01"
end_date = "2026-05-31"

ma_window = 60
retry_wait = 60
rate_limit_wait = 3660
rebuild_signal = False

# Fetch extra daily history so the first target date can have a valid MA60.
daily_start_date = (
    pd.Timestamp(start_date) - pd.Timedelta(days=ma_window * 3)
).strftime("%Y-%m-%d")

data_dir = Path("D:/Github/note/data")
stock_universe_output = data_dir / "stock_universe.parquet"
daily_output = data_dir / "daily_stock_price.parquet"
full_kbar_dir = data_dir / "kbar_data"
signal_output = full_kbar_dir / "above_ma60_signal.parquet"
kbar_output_dir = full_kbar_dir / "above_ma60"
kbar_output_dir.mkdir(parents=True, exist_ok=True)

### get daily data

In [ ]:
# 1. Load or refresh raw daily prices for valid stocks only.
stock_ids = get_valid_stock_ids(client, output_file=stock_universe_output)
print(f"Valid stock count: {len(stock_ids):,}")

daily = load_or_refresh_daily_prices(
    client=client,
    output_file=daily_output,
    start_date=daily_start_date,
    end_date=end_date,
    stock_ids=stock_ids,
    retry_wait=retry_wait,
)

daily["date"] = pd.to_datetime(daily["date"])
print(f"Daily rows: {len(daily):,}")
print(f"Daily dates: {daily['date'].nunique():,}")
print(f"Daily stocks: {daily['stock_id'].astype(str).nunique():,}")
print(f"Daily range: {daily['date'].min().date()} ~ {daily['date'].max().date()}")

In [ ]:
# Optional: compact the old huge daily_price.parquet without calling API.
old_daily_output = data_dir / "daily_price.parquet"

if not old_daily_output.exists():
    raise FileNotFoundError(old_daily_output)

if stock_universe_output.exists():
    stock_ids_for_compact = pd.read_parquet(stock_universe_output)["stock_id"].astype(str).tolist()
else:
    stock_ids_for_compact = infer_stock_ids_from_kbar_dir(full_kbar_dir)
    if not stock_ids_for_compact:
        raise FileNotFoundError(
            "No stock_universe.parquet and no kbar_data parquet files to infer stock ids from."
        )
    pd.DataFrame({"stock_id": stock_ids_for_compact}).to_parquet(
        stock_universe_output,
        index=False,
    )

print(f"Compacting {len(stock_ids_for_compact):,} stock ids")

daily = compact_daily_price_parquet(
    source_file=old_daily_output,
    output_file=daily_output,
    stock_ids=stock_ids_for_compact,
)

print(f"Daily rows: {len(daily):,}")
print(f"Daily dates: {daily['date'].nunique():,}")
print(f"Daily stocks: {daily['stock_id'].astype(str).nunique():,}")
print(f"Daily range: {daily['date'].min().date()} ~ {daily['date'].max().date()}")

### build 60ma signal

In [ ]:
# 2. Build/load date x stock_id signal matrix.
# True means this stock passed yesterday's close > MA60 condition for today's kbar.
signal = load_or_build_above_ma_signal(
    daily=daily,
    output_file=signal_output,
    start_date=start_date,
    end_date=end_date,
    ma_window=ma_window,
    rebuild=rebuild_signal,
)

print(f"Signal shape: {signal.shape[0]} dates x {signal.shape[1]} stocks")
print(f"Signal true cells: {int(signal.to_numpy().sum())}")

### get minute kbar data

In [ ]:
signal = pd.read_parquet(signal_output)
signal.index = pd.to_datetime(signal.index)

# 3. Build filtered K bars. Prefer local full-market parquet; call API only for missing days/ids.
for trade_date, signal_row in tqdm(signal.iterrows(), total=len(signal), desc="Build filtered kbar"):
    stock_ids = signal_row_to_stock_ids(signal_row)
    if not stock_ids:
        continue

    date_str = pd.Timestamp(trade_date).strftime("%Y-%m-%d")
    out_file = kbar_output_dir / f"{date_str}.parquet"
    expected_ids = set(stock_ids)

    existing_ids = read_kbar_ids(out_file)
    if expected_ids.issubset(existing_ids):
        continue

    if write_filtered_from_full_kbar(
        date_str=date_str,
        stock_ids=stock_ids,
        full_kbar_dir=full_kbar_dir,
        output_dir=kbar_output_dir,
    ):
        continue

    try:
        ok = client.get_multi_kbar(
            start_date=date_str,
            end_date=date_str,
            output_dir=kbar_output_dir,
            stock_ids=stock_ids,
            max_retries=3,
            retry_wait=retry_wait,
            rate_limit_wait=rate_limit_wait,
        )
        if ok is False:
            print(f"Stopped at {date_str}; rerun this cell later to resume.")
            break
    except Exception as exc:
        print(f"{date_str} failed: {exc}")
        time.sleep(retry_wait)

Build filtered kbar:   0%|          | 0/1065 [00:00<?, ?it/s]

股票數: 1575
取得交易日列表...
交易日數: 1
已完成: 0 天 / 待下載: 1 天
預估剩餘: 0.0 小時


In [ ]:
# Optional inspection
signal = pd.read_parquet(signal_output)
signal.index = pd.to_datetime(signal.index)
signal

stock_id,00400A,00401A,00403A,0050,0051,0052,0053,0054,0055,0056,...,9944,9945,9946,9949,9950,9951,9955,9958,9960,9962
date,,,,,,,,,,,,,,,,,,,,,
2022-01-03,False,False,False,True,True,True,True,True,True,True,...,True,False,False,True,True,False,True,False,True,True
2022-01-04,False,False,False,True,True,True,True,True,True,True,...,True,False,False,True,True,True,True,True,True,True
2022-01-05,False,False,False,True,True,True,True,True,True,True,...,True,False,False,True,True,True,True,True,True,True
2022-01-06,False,False,False,True,True,True,True,True,True,True,...,True,False,False,True,True,False,True,True,True,True
2022-01-07,False,False,False,True,True,True,True,True,True,True,...,True,False,False,True,True,False,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-25,False,False,False,True,True,True,True,False,True,True,...,True,False,False,False,True,False,False,False,True,False
2026-05-26,False,False,False,True,True,True,True,False,True,True,...,True,False,False,False,False,False,False,False,True,False
2026-05-27,False,False,False,True,True,True,True,False,True,True,...,True,False,False,False,False,False,False,False,True,False
